# Assignment 7: Data Cleaning and Preprocessing

| | |
|---|---|
| **Student** | Lokesh Patwal |
| **Programme** | Yuva Intern - Data Science Fundamentals Assessment |
| **Assignment** | 7 of 10 |

## 2. Aim
To clean a raw, messy table step by step: find missing values, remove duplicates, fix inconsistent text, convert data types, detect outliers and save a clean dataset ready for analysis.

## 3. Objectives
1. Import a raw dataset and audit its problems.
2. Identify and treat missing values.
3. Remove duplicate records.
4. Standardise inconsistent text.
5. Convert text to numbers and dates.
6. Detect outliers and invalid values.
7. Save the clean dataset and report before/after counts.

## 4. Introduction and Theory
Real data is rarely clean. Typical problems: **missing values**, **duplicates**, **inconsistent labels** ("Male", "male ", "M"), **wrong types** (numbers stored as text), **impossible values** (age 250) and **outliers**.

Golden rules: (1) never edit the raw file, work on a copy; (2) audit *before* cleaning; (3) record every change with counts; (4) do not delete data just because it is unusual: decide using domain knowledge.

Methods used: `drop_duplicates`, string cleaning, `pd.to_numeric(errors="coerce")`, `pd.to_datetime`, a *domain rule* (marks must be 0-100), the **IQR rule**, and **median imputation** (median is not pulled by extreme values).

## 5. Software and Libraries Required
Python 3, pandas, NumPy, Matplotlib, and the project's own `src/ds_helpers.py`.

## 6. Dataset Details
**IMPORTANT - this dataset is SYNTHETIC (artificial).** No suitable real "dirty" public file could be downloaded in the working environment, so a small student-records table is generated by code with a fixed random seed (42). Problems are injected on purpose and listed below, so the cleaning result can be *checked against the known truth*. It contains no real person's data.

| Column | Meaning |
|---|---|
| student_id | ID such as S001 |
| name | first name (random letters pattern, not real people) |
| gender | text with inconsistent spellings |
| age | years, stored as text in some rows |
| city | text with spaces / capitalisation problems |
| marks | exam marks out of 100, with text and impossible values |
| attendance_pct | attendance percentage, some missing |
| join_date | date in two formats, a few invalid |

Injected problems: 5 duplicate rows, inconsistent gender and city spellings, `N/A` text in marks, marks of 950 and -5, missing attendance and age, an age of 250, and invalid dates. The file is saved as `data/raw/students_messy_synthetic.csv`.

## 7. Step-by-Step Procedure
1. Generate/import the raw data. 2. Audit. 3. Remove duplicates. 4. Fix text. 5. Convert types. 6. Handle invalid values and outliers. 7. Fill missing values. 8. Compare before/after. 9. Save.

## 8. Complete Python Code and 9. Expected Output
The output under each code cell is the actual output produced when this notebook was executed.

### Step 1: Create / import the raw data (synthetic, seeded)

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))     # lets the notebook import src/ds_helpers.py
from src import ds_helpers as h

RAW = Path("../data/raw/students_messy_synthetic.csv")
RAW.parent.mkdir(parents=True, exist_ok=True)

def make_messy_students(seed=42, n=80):
    """Create a SYNTHETIC student table with deliberately injected data problems."""
    rng = np.random.default_rng(seed)
    syllables = ["ra", "mi", "ka", "so", "ve", "lu", "an", "ti", "pa", "no"]
    names = ["".join(rng.choice(syllables, 2)).title() + str(i) for i in range(n)]
    df = pd.DataFrame({
        "student_id": [f"S{i + 1:03d}" for i in range(n)],
        "name": names,
        "gender": rng.choice(["Male", "Female"], n),
        "age": rng.integers(18, 26, n).astype(object),
        "city": rng.choice(["Jaipur", "Delhi", "Mumbai", "Pune"], n),
        "marks": rng.integers(35, 100, n).astype(object),
        "attendance_pct": rng.integers(60, 100, n).astype(float),
        "join_date": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 200, n), unit="D"),
    })
    df["join_date"] = df["join_date"].dt.strftime("%Y-%m-%d")

    # ---- injected problems (fixed positions so the checks below are exact) ----
    df.loc[[3, 10, 17, 24], "gender"] = ["male", "F", "FEMALE ", " m"]
    df.loc[[2, 8, 15, 30, 41], "city"] = [" jaipur", "DELHI", "Mumbai ", "pune", "jaipur  "]
    df.loc[[5, 22, 44], "marks"] = ["N/A", "seventy", "N/A"]
    df.loc[[7, 33], "marks"] = [950, -5]
    df.loc[[12, 29, 51, 60], "attendance_pct"] = np.nan
    df.loc[[19, 55], "age"] = [None, None]
    df.loc[[26], "age"] = [250]
    df.loc[[4, 37], "age"] = ["21", "22"]                 # numbers stored as text
    df.loc[[6, 18, 40], "join_date"] = ["15/03/2026", "2026-02-31", "not recorded"]
    df = pd.concat([df, df.iloc[[0, 9, 20, 35, 70]]], ignore_index=True)   # 5 exact duplicates
    return df

raw = make_messy_students()
raw.to_csv(RAW, index=False)
print("Saved raw file:", RAW.name, "| shape:", raw.shape)
display(raw.head(8))

Saved raw file: students_messy_synthetic.csv | shape: (85, 8)


,student_id,name,gender,age,city,marks,attendance_pct,join_date
0,S001,Rati0,Female,19,Delhi,99,71.0,2026-04-18
1,S002,Anve1,Female,22,Jaipur,85,93.0,2026-01-09
2,S003,Vepa2,Male,18,jaipur,55,70.0,2026-04-09
3,S004,Raan3,male,23,Pune,98,91.0,2026-02-06
4,S005,Kara4,Male,21,Pune,67,78.0,2026-06-27
5,S006,Luno5,Female,18,Jaipur,N/A,69.0,2026-02-17
6,S007,Titi6,Male,24,Jaipur,63,73.0,15/03/2026
7,S008,Titi7,Male,21,Jaipur,950,81.0,2026-02-19


### Step 2: Audit the raw data before changing anything

In [2]:
df = pd.read_csv(RAW)                                 # work on a fresh copy read from the raw file
before = {"rows": len(df), "missing_cells": int(df.isna().sum().sum()), "duplicates": int(df.duplicated().sum())}
print("Data types as read from the file:")
print(df.dtypes.to_string())
print()
print("Missing-value report:")
display(h.missing_report(df))
print("Duplicate rows:", before["duplicates"])
print("Gender spellings:", sorted(df["gender"].unique()))
print("City spellings  :", sorted(df["city"].unique()))

Data types as read from the file:
student_id            str
name                  str
gender                str
age               float64
city                  str
marks                 str
attendance_pct    float64
join_date             str

Missing-value report:


,missing,percent
attendance_pct,4,4.71
age,2,2.35
marks,2,2.35
student_id,0,0.00
gender,0,0.00
name,0,0.00
city,0,0.00
join_date,0,0.00


Duplicate rows: 5
Gender spellings: [' m', 'F', 'FEMALE ', 'Female', 'Male', 'male']
City spellings  : [' jaipur', 'DELHI', 'Delhi', 'Jaipur', 'Mumbai', 'Mumbai ', 'Pune', 'jaipur  ', 'pune']


### Step 3: Remove duplicate records

In [3]:
df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before['rows']} -> {len(df)}  ({before['rows'] - len(df)} duplicates removed)")
print("Duplicate student_ids left:", int(df['student_id'].duplicated().sum()))

Rows: 85 -> 80  (5 duplicates removed)
Duplicate student_ids left: 0


### Step 4: Fix inconsistent text
`clean_text` strips spaces and fixes capitalisation. Short codes (M, F) are mapped to full words with a dictionary.

In [4]:
df["city"] = h.clean_text(df["city"])
df["gender"] = h.clean_text(df["gender"]).replace({"M": "Male", "F": "Female"})
print("Gender now:", sorted(df["gender"].unique()))
print("City now  :", sorted(df["city"].unique()))
print(df["gender"].value_counts().to_string())

Gender now: ['Female', 'Male']
City now  : ['Delhi', 'Jaipur', 'Mumbai', 'Pune']
gender
Male      43
Female    37


### Step 5: Convert data types

In [5]:
for column in ["age", "marks"]:
    df[column], failed = h.to_numeric_with_report(df[column])
    print(f"{column}: converted to numbers, {failed} text value(s) could not be converted and became NaN")

# Dates: two accepted formats; anything else becomes NaT (missing date)
iso = pd.to_datetime(df["join_date"], format="%Y-%m-%d", errors="coerce")
day_first = pd.to_datetime(df["join_date"], format="%d/%m/%Y", errors="coerce")
df["join_date"] = iso.fillna(day_first)
print("Invalid dates (now NaT):", int(df["join_date"].isna().sum()))
print(df.dtypes.to_string())

age: converted to numbers, 0 text value(s) could not be converted and became NaN
marks: converted to numbers, 1 text value(s) could not be converted and became NaN
Invalid dates (now NaT): 2
student_id                   str
name                         str
gender                    string
age                      float64
city                      string
marks                    float64
attendance_pct           float64
join_date         datetime64[us]


### Step 6: Invalid values and outliers
First a **domain rule**: marks must be 0-100 and age must be 15-60, so anything outside is an *entry error* and is set to missing. Then the **IQR rule** checks what is left for statistical outliers.

In [6]:
bad_marks = ~df["marks"].between(0, 100) & df["marks"].notna()
bad_age = ~df["age"].between(15, 60) & df["age"].notna()
print("Impossible marks:", df.loc[bad_marks, "marks"].tolist())
print("Impossible age  :", df.loc[bad_age, "age"].tolist())
df.loc[bad_marks, "marks"] = np.nan
df.loc[bad_age, "age"] = np.nan

for column in ["marks", "age", "attendance_pct"]:
    lower, upper = h.iqr_bounds(df[column])
    flagged = int(h.iqr_outlier_mask(df[column]).sum())
    print(f"{column:<15} IQR fences [{lower:.1f}, {upper:.1f}] -> {flagged} statistical outlier(s)")

Impossible marks: [950.0, -5.0]
Impossible age  : [250.0]
marks           IQR fences [-12.0, 144.0] -> 0 statistical outlier(s)
age             IQR fences [13.0, 29.0] -> 0 statistical outlier(s)
attendance_pct  IQR fences [37.0, 125.0] -> 0 statistical outlier(s)


### Step 7: Handle the remaining missing values
We fill numeric gaps with the **median** and keep a True/False *flag column* so the filled rows can always be identified. Missing dates are **not** invented; they stay missing.

In [7]:
for column in ["marks", "age", "attendance_pct"]:
    df[f"{column}_was_missing"] = df[column].isna()
    fill_value = df[column].median()
    df[column] = df[column].fillna(fill_value)
    print(f"{column:<15}: filled {int(df[f'{column}_was_missing'].sum())} gap(s) with median {fill_value}")

print("Missing cells left (only dates):", int(df.drop(columns=[c for c in df if c.endswith('_was_missing')]).isna().sum().sum()))

marks          : filled 5 gap(s) with median 67.0
age            : filled 3 gap(s) with median 21.0
attendance_pct : filled 4 gap(s) with median 80.0
Missing cells left (only dates): 2


### Step 8: Before / after comparison and saving

In [8]:
after = {"rows": len(df), "missing_cells": int(df[["name", "gender", "age", "city", "marks", "attendance_pct", "join_date"]].isna().sum().sum()), "duplicates": int(df.duplicated().sum())}
comparison = pd.DataFrame({"before": before, "after": after})
display(comparison)

OUT = Path("../data/processed/students_clean.csv")
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False)
print("Saved clean file:", OUT.name, "| shape:", df.shape)
display(df.describe().round(2))

,before,after
rows,85,80
missing_cells,8,2
duplicates,5,0


Saved clean file: students_clean.csv | shape: (80, 11)


,age,marks,attendance_pct,join_date
count,80.00,80.00,80.00,78
mean,21.15,67.51,80.28,2026-04-09 04:36:55.384615
min,18.00,35.00,60.00,2026-01-02 00:00:00
25%,19.00,48.50,70.75,2026-02-20 00:00:00
50%,21.00,67.00,80.00,2026-04-04 12:00:00
75%,23.00,83.50,92.00,2026-06-03 00:00:00
max,25.00,99.00,99.00,2026-07-15 00:00:00
std,2.16,19.92,11.51,NaN


### Step 9: Validation checks (against the known injected problems)

In [9]:
assert before["rows"] == 85 and before["duplicates"] == 5
assert len(df) == 80 and df["student_id"].is_unique
assert sorted(df["gender"].unique()) == ["Female", "Male"]
assert sorted(df["city"].unique()) == ["Delhi", "Jaipur", "Mumbai", "Pune"]
assert df["marks"].between(0, 100).all() and df["age"].between(15, 60).all()
assert df["marks_was_missing"].sum() == 5          # N/A x2, 'seventy', 950, -5 (N/A at row 44 is one of the 3 text values)
assert df["age_was_missing"].sum() == 3            # 2 blanks + the impossible age 250
assert pd.api.types.is_numeric_dtype(df["marks"]) and pd.api.types.is_datetime64_any_dtype(df["join_date"])
print("All validation checks passed: the cleaning matches the injected problems exactly.")

All validation checks passed: the cleaning matches the injected problems exactly.


## 10. Output Explanation
* **Audit:** the raw file had `85` rows, `5` duplicates and `8` missing cells. The marks column was read as text because one entry is the word `seventy`, so it could not be averaged directly; age was read as numbers but contains the impossible value 250.
* **Duplicates:** removing exact repeated rows reduced the table from `85` to `80` rows; every `student_id` is now unique.
* **Text:** gender went from several spellings (`male`, `F`, `FEMALE `, ` m` ...) to just *Female* and *Male*; city spellings were reduced to 4 clean names.
* **Types:** `to_numeric(errors="coerce")` turns unreadable text into `NaN` instead of crashing; the dates were parsed from two formats and `2` unreadable dates remain missing.
* **Invalid values:** marks of 950 and -5 and an age of 250 are impossible, so they were treated as entry errors. After that the IQR rule flagged `0` marks outliers.
* **Missing values:** gaps in marks, age and attendance were filled with the median and flagged. After cleaning, missing cells in the data columns fell from `8` to `2` (the remaining ones are unreadable dates, deliberately not invented).

## 11. Observations and Analysis
* Each cleaning step changed the data in a way that could be counted and checked. Because the data is synthetic, we could prove that nothing was missed.
* The domain rule (marks 0-100) caught errors that the IQR rule alone would treat differently: an outlier is a statistical idea, an *impossible* value is a logical one.
* Median imputation keeps the centre of the data stable but reduces variation. A flag column preserves the information that a value was filled.
* Dropping duplicates assumes an exact repeat is an error. In real data (for example repeat purchases) identical rows can be genuine, so always check first.
* Every step was done on a copy; the raw CSV was never edited.

**Common mistakes**
| Mistake | Fix |
|---|---|
| Cleaning the raw file in place | Always keep raw data untouched in `data/raw/` |
| Filling all missing values with 0 | Use median/mean/mode, or leave missing if unknown |
| Removing every outlier automatically | Check whether it is an error or a real extreme |
| Converting types without checking failures | Count how many conversions failed |

**Practice tasks**
1. Instead of the median, fill attendance with the mean of each city and compare the results.
2. Add a column `result` that is "Pass" when marks >= 40.
3. Count how many students joined in each month.

## 12. Result / Conclusion
A messy synthetic student table (85 rows, many injected problems) was cleaned into an 80-row, correctly typed, consistent dataset saved in `data/processed/`. All validation checks passed, which shows that the cleaning logic does what it claims. Documenting each step with before/after counts makes the process transparent and repeatable.